In [ ]:
import networkx as nx
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
G = nx.Graph()
G.add_node(1, text="I absolutely love this product, it's amazing!")
G.add_node(2, text="This is the worst service I have ever had.")
G.add_node(3, text="It's okay, but could be much better.")
G.add_edge(1, 2)
G.add_edge(2, 3)


In [ ]:
vocab_size = 1000
max_length = 20
embedding_dim = 16


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
    tf.keras.layers.SpatialDropout1D(0.2),
    tf.keras.layers.GRU(32), # RNN Variant: Gated Recurrent Unit
    tf.keras.layers.Dense(1, activation='sigmoid')
])


In [ ]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
all_text = [data['text'] for node, data in G.nodes(data=True)]
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(all_text)
sequences = tokenizer.texts_to_sequences(all_text)
padded = pad_sequences(sequences, maxlen=max_length, padding='post')
# Generate Predictions (0 = Negative, 1 = Positive)
predictions = model.predict(padded)


In [ ]:
for i, node in enumerate(G.nodes()):
    sentiment_score = float(predictions[i])
    G.nodes[node]['sentiment'] = "Positive" if sentiment_score > 0.5 else "Negative"
    G.nodes[node]['score'] = sentiment_score

In [ ]:
# Display Results
for node, data in G.nodes(data=True):
    print(f"Node {node} | Sentiment: {data['sentiment']} ({data['score']:.2f})")